# Optuna Results Viewer
PyCharmのこのセルを実行すると、Optunaのデータベースから結果を取得し、インタラクティブなテーブルとして表示します。
`target_study_id` または `target_study_name` を変更することで、動的に読み込むStudyを切り替えられます。

In [16]:
import optuna
import pandas as pd
import sqlite3

# ==========================================
# 設定: ここを書き換えて動的にStudyを指定します
# ==========================================
db_path = "optimize_rc.db"

# Studyの指定方法（どちらかを使用。両方Noneなら一番最新を取得します）
target_study_id = 18    # 例: 1 (データベース内のIDで指定する場合)
target_study_name = None  # 例: "qrc_lorenz_vpt..." (名前で指定する場合)

# フィルタリング設定: value（VPTなど）がこの値以上の行のみ表示します
min_value_threshold = 4.0 # 例: 5.0
# ==========================================

storage_url = f"sqlite:///{db_path}"

# 利用可能なStudy一覧を取得するヘルパー関数
def get_studies(db_file):
    try:
        with sqlite3.connect(db_file) as conn:
            cursor = conn.cursor()
            cursor.execute("SELECT study_id, study_name FROM studies ORDER BY study_id ASC")
            return cursor.fetchall()
    except Exception as e:
        return []

studies = get_studies(db_path)

if not studies:
    print(f"データベース '{db_path}' にStudyが見つからない、またはファイルが存在しません。")
    df_view = pd.DataFrame({"Error": ["Studyが見つかりません"]})
else:
    print("【利用可能なStudy一覧】")
    for s_id, s_name in studies:
        print(f"  ID: {s_id} | Name: {s_name}")
        
    # どのStudyを読み込むか決定
    selected_name = None
    if target_study_id is not None:
        for s_id, s_name in studies:
            if s_id == target_study_id:
                selected_name = s_name
                break
        if selected_name is None:
            print(f"\n⚠️ 警告: 指定された ID '{target_study_id}' は見つかりませんでした。")
            
    elif target_study_name is not None:
        for s_id, s_name in studies:
            if s_name == target_study_name:
                selected_name = s_name
                break
        if selected_name is None:
            print(f"\n⚠️ 警告: 指定された Name '{target_study_name}' は見つかりませんでした。")

    # 指定がない、または見つからなかった場合は最新（最後）のものを選択
    if selected_name is None:
        selected_name = studies[-1][1]
        print(f"\n-> 🔄 最新のStudyを読み込みます: {selected_name}")
    else:
        print(f"\n-> ✅ 指定されたStudyを読み込みます: {selected_name}")

    # Optunaからデータを取得
    study = optuna.load_study(study_name=selected_name, storage=storage_url)
    df = study.trials_dataframe()
    
    # 見やすいようにフォーマット（COMPLETEのみ抽出し、VPTで降順ソート）
    if 'state' in df.columns:
        df = df[df['state'] == 'COMPLETE']
    if 'value' in df.columns:
        # 指定された閾値以上のものだけをフィルタリング
        if min_value_threshold is not None:
            df = df[df['value'] >= min_value_threshold]
        df = df.sort_values(by='value', ascending=False)
        
    # 不要な列を隠して見やすくする（datetime等）
    drop_cols = [c for c in df.columns if c.startswith('datetime_') or c.startswith('duration')]
    df_view = df.drop(columns=drop_cols)
    
    # 必要な列を抽出してCSVに保存
    import os
    # 'value' と 'params_' で始まるすべての列を抽出
    export_cols = ['value'] + [c for c in df_view.columns if c.startswith('params_')]
    valid_export_cols = [c for c in export_cols if c in df_view.columns]
    
    if valid_export_cols:
        out_path = os.path.join("/home/yoshi/PycharmProjects/Reservoir/benchmarks", "filtered_optuna_results.csv")
        df_view[valid_export_cols].to_csv(out_path, index=False)
        print(f"\n✅ {len(valid_export_cols)-1} 個のパラメータを含むデータを '{out_path}' に保存しました。")

# DataFrameを表示（PyCharmが綺麗なテーブルでレンダリングします）
df_view.head(50)

【利用可能なStudy一覧】
  ID: 4 | Name: optimize_rc_MACKEY_GLASS_Standard_Random100_ridge
  ID: 5 | Name: optimize_rc_MACKEY_GLASS_Standard_Random400_ridge
  ID: 6 | Name: optimize_rc_LORENZ_Standard_Random30_ridge
  ID: 7 | Name: optimize_rc_MNIST_Standard_Random100_ridge
  ID: 9 | Name: optimize_rc_MNIST_Min-1Max1_Random100_ridge
  ID: 10 | Name: optimize_rc_MNIST_Min0Max1_Random100_ridge
  ID: 11 | Name: optimize_rc_MNIST_MinMax_Random100_ridge
  ID: 12 | Name: optimize_rc_MACKEY_GLASS_Standard_Random100_ridge_kai
  ID: 15 | Name: optimize_rc_MNIST_MinMax_Random100_ridge_kai2
  ID: 16 | Name: optimize_rc_MNIST_MinMax_Random1200_ridge_kai2
  ID: 17 | Name: optimize_rc_MACKEY_GLASS_MinMax_Random100_ridge_kai2
  ID: 18 | Name: optimize_rc_MACKEY_GLASS_MinMax_Random64_ridge_kai2
  ID: 19 | Name: optimize_rc_MACKEY_GLASS_MinMax_Random1024_ridge_kai2
  ID: 20 | Name: optimize_rc_MNIST_BoundedAffineScaler_Random100_ridge_kai3

-> ✅ 指定されたStudyを読み込みます: optimize_rc_MACKEY_GLASS_MinMax_Random64_ridge_k

,number,value,params_bias_scale,params_feature_max,params_feature_min,params_input_connectivity,params_input_scale,params_leak_rate,params_rc_connectivity,params_spectral_radius,...,user_attrs_status,user_attrs_truth_max,user_attrs_truth_mean,user_attrs_truth_min,user_attrs_truth_std,user_attrs_var_ratio,user_attrs_vpt_lt,user_attrs_vpt_steps,user_attrs_vpt_threshold,state
1106,1106,6.674699,0.676372,0.197144,-1.252115,0.486849,0.780619,0.553195,0.983015,1.198058,...,success,NaN,NaN,NaN,NaN,0.990843,6.674699,1108.0,0.4,COMPLETE
1411,1411,5.180723,0.308620,1.072885,-1.324971,0.495044,0.818340,0.285867,0.752732,0.917086,...,success,NaN,NaN,NaN,NaN,0.992046,5.180723,860.0,0.4,COMPLETE
1338,1338,4.987952,0.597646,0.857688,-0.894604,0.554170,0.998556,0.292697,0.808629,0.813785,...,success,NaN,NaN,NaN,NaN,0.997487,4.987952,828.0,0.4,COMPLETE
939,939,4.662651,0.684441,0.504870,-0.939013,0.475384,0.909089,0.587090,0.981418,1.123375,...,success,NaN,NaN,NaN,NaN,0.996749,4.662651,774.0,0.4,COMPLETE
1306,1306,4.650602,0.626130,0.522263,-1.035302,0.495945,0.812931,0.563577,0.964304,1.002756,...,success,NaN,NaN,NaN,NaN,0.994307,4.650602,772.0,0.4,COMPLETE
980,980,4.644578,0.688977,0.143420,-1.190388,0.445994,0.826086,0.557973,0.974202,1.112689,...,success,NaN,NaN,NaN,NaN,1.000720,4.644578,771.0,0.4,COMPLETE
1727,1727,4.626506,0.604803,0.649589,-1.731653,0.449539,0.966798,0.625206,0.926694,0.917665,...,success,NaN,NaN,NaN,NaN,1.002620,4.626506,768.0,0.4,COMPLETE
2000,2000,4.445783,0.698316,0.840515,-0.970558,0.510114,0.892295,0.294944,0.969205,1.056646,...,success,NaN,NaN,NaN,NaN,0.991236,4.445783,738.0,0.4,COMPLETE
